Part 1 — Warm-Up Tutorial: User-Based Recommender

In [1]:
# Step 1 - Import the libraries
import pandas as pd
import numpy as np

# Step 2 - Load the dataset into a dataframe
ratings = pd.read_csv('ccai422_lab03_part1_data.csv')
ratings = ratings[["user_id", "movie_id", "rating"]]

# Step 3 - Explore the data
print('The number of data points in this dataset: ' + str(len(ratings)))
print('The number of items (i.e. movies) in the dataset: ' + str(ratings['movie_id'].nunique()))
print('The number of users in the dataset: ' + str(ratings['user_id'].nunique()))

ratings_per_users = ratings.groupby('user_id').count()
print('The average ratings per user: ' + str(round(ratings_per_users.mean()[0], 2)))
print('The below table shows the number of ratings per user\n')
print(ratings_per_users)

# Step 4 - Build the rating matrix
r_matrix = ratings.pivot_table(values='rating', index='user_id', columns='movie_id')

r_matrix_dummy = r_matrix.copy()
r_matrix_dummy = r_matrix_dummy.rename_axis('user_id', axis=1).rename_axis(None, axis=0)
r_matrix_dummy = r_matrix_dummy.fillna(0)
print(r_matrix_dummy.head())

# Step 5 - Compute the correlation between users
users_rating_matrix = r_matrix_dummy.T
pearson_sim = users_rating_matrix.corr()

# Step 6 - Retrieve the prediction variables
userX = 5
rXmean = r_matrix[userX].mean()
n = 2
topn = pearson_sim[[userX]].nlargest(n + 1, userX).index.tolist()[1:]
neighbors_sim = pearson_sim[[userX]].nlargest(n + 1, userX)[1:]
r_matrix_topn = r_matrix[topn]
neighbors_means = r_matrix_topn.mean()
averaged_neighbors_ratings = r_matrix_topn.sub(neighbors_means, axis=1)

# Step 7 - Find the unrated items for the target user
unrated_target = r_matrix[r_matrix[userX].isna()][topn]
unrated_target = unrated_target.rename_axis('movie_id', axis=1).rename_axis(None, axis=0)
unrated_target.dropna(axis=0, how='all', inplace=True)
print(unrated_target.head())

# Step 8 - Predict the rating
itemX = 7
predicted_value = rXmean + (
    (neighbors_sim.T.dot(averaged_neighbors_ratings.loc[itemX].T).values[0])
    / neighbors_sim.sum()
)
print("Predicted rating:", predicted_value)


The number of data points in this dataset: 100000
The number of items (i.e. movies) in the dataset: 1682
The number of users in the dataset: 943
The average ratings per user: 106.04
The below table shows the number of ratings per user

         movie_id  rating
user_id                  
1             272     272
2              62      62
3              54      54
4              24      24
5             175     175
...           ...     ...
939            49      49
940           107     107
941            22      22
942            79      79
943           168     168

[943 rows x 2 columns]
user_id  1     2     3     4     5     6     7     8     9     10    ...  \
1         5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2         4.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   2.0  ...   
3         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5         4.0   3.0   

/tmp/ipykernel_7215/1190339116.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print('The average ratings per user: ' + str(round(ratings_per_users.mean()[0], 2)))


movie_id  307  22 
2         3.0  NaN
3         3.0  NaN
6         NaN  3.0
7         5.0  5.0
8         NaN  5.0
Predicted rating: 5    4.478432
dtype: float64


Part 2 — Build a User-Based Recommender with Pandas

In [2]:
# Step 1 - Import the libraries
import pandas as pd
import numpy as np

# Step 2 - Upload and inspect the dataset

# Step 3 - Load the data into a dataframe
user_item_df = pd.read_csv('ccai422_lab03_part2_data.csv', index_col=0)

# Step 4 - Display the dataframe
print(user_item_df.head())

# Step 5 - Compute the Pearson correlation with pandas

# Step 6 - Store the correlation result
user_item_corr = user_item_df.corr()
print(user_item_corr.head())

# Step 7 - Identify Alice's neighbors
n = 2
topn = user_item_corr[['Alice']].nlargest(n + 1, 'Alice').index.tolist()[1:]
neighbors_sim = user_item_corr[['Alice']].nlargest(n + 1, 'Alice')[1:]
print("Alice's top", n, "neighbors:", topn)
print(neighbors_sim)

# Step 8 - Prepare the prediction variables
ra = user_item_df['Alice'].mean()
target_item = 'item5'
r_bp = user_item_df.loc[target_item, topn]      # neighbors' ratings for Item5
rb = user_item_df[topn].mean()                  # neighbors' mean ratings

# Step 9 - Apply the prediction formula
numerator = (neighbors_sim['Alice'].values * (r_bp.values - rb.values)).sum()
denominator = neighbors_sim['Alice'].abs().sum()
predicted_alice_item5 = ra + (numerator / denominator)

# Step 10 - Print and verify the prediction
print("Predicted rating for Alice on Item5:", predicted_alice_item5)


       Alice  User1  User2  User3  User4
item1    5.0      3      4      3      1
item2    3.0      1      3      3      5
item3    4.0      2      4      1      5
item4    4.0      3      3      5      2
item5    NaN      3      5      4      1
          Alice     User1     User2     User3     User4
Alice  1.000000  0.852803  0.707107  0.000000 -0.792118
User1  0.852803  1.000000  0.467707  0.489956 -0.900149
User2  0.707107  0.467707  1.000000 -0.161165 -0.466569
User3  0.000000  0.489956 -0.161165  1.000000 -0.641503
User4 -0.792118 -0.900149 -0.466569 -0.641503  1.000000
Alice's top 2 neighbors: ['User1', 'User2']
          Alice
User1  0.852803
User2  0.707107
Predicted rating for Alice on Item5: 4.871979899370592


Part 3 — Individual Assessment: Pearson Correlation from Scratch

In [3]:
# Step 1 - Import the libraries
import pandas as pd
import numpy as np

# Step 2 - Load the same data used in Part 2
user_item_df = pd.read_csv('ccai422_lab03_part2_data.csv', index_col=0)
user_item_df = user_item_df.loc[:, ~user_item_df.columns.duplicated()]
users = user_item_df.columns.tolist()

# Task 1 - Implement the Pearson correlation
def pearson_corr(a, b, df):
    """
    sim(a,b) = sum((r_a,p - mean_a)(r_b,p - mean_b)) /
               sqrt( sum((r_a,p - mean_a)^2) * sum((r_b,p - mean_b)^2) )
    Only uses items both users rated (drops NaNs).
    """
    if a == b:
        return 1.0

    pair = df[[a, b]].dropna()
    ra_mean = df[a].mean()
    rb_mean = df[b].mean()

    num = ((pair[a] - ra_mean) * (pair[b] - rb_mean)).sum()
    den = np.sqrt(((pair[a] - ra_mean) ** 2).sum()) * np.sqrt(((pair[b] - rb_mean) ** 2).sum())

    return num / den if den != 0 else 0.0

# Build the full user-to-user similarity matrix manually
manual_corr = pd.DataFrame(index=users, columns=users, dtype=float)
for a in users:
    for b in users:
        manual_corr.loc[a, b] = pearson_corr(a, b, user_item_df)

print(manual_corr)

# Task 2 - Predict Alice's Item5 rating
n = 2
topn = manual_corr[['Alice']].nlargest(n + 1, 'Alice').index.tolist()[1:]
neighbors_sim = manual_corr[['Alice']].nlargest(n + 1, 'Alice')[1:]

ra = user_item_df['Alice'].mean()
target_item = 'item5'
r_bp = user_item_df.loc[target_item, topn]
rb = user_item_df[topn].mean()

numerator = (neighbors_sim['Alice'].values * (r_bp.values - rb.values)).sum()
denominator = neighbors_sim['Alice'].abs().sum()
predicted_alice_item5_manual = ra + (numerator / denominator)

print("Predicted rating for Alice on Item5 (manual Pearson):", predicted_alice_item5_manual)

# Task 3 - Confirm the result

pandas_corr = user_item_df.corr()
topn_pd = pandas_corr[['Alice']].nlargest(n + 1, 'Alice').index.tolist()[1:]
neighbors_sim_pd = pandas_corr[['Alice']].nlargest(n + 1, 'Alice')[1:]
r_bp_pd = user_item_df.loc[target_item, topn_pd]
rb_pd = user_item_df[topn_pd].mean()
num_pd = (neighbors_sim_pd['Alice'].values * (r_bp_pd.values - rb_pd.values)).sum()
den_pd = neighbors_sim_pd['Alice'].abs().sum()
predicted_alice_item5_pandas = ra + (num_pd / den_pd)

print("Part 2 (pandas) prediction:", predicted_alice_item5_pandas)
print("Part 3 (manual) prediction:", predicted_alice_item5_manual)
print("Match:", np.isclose(predicted_alice_item5_pandas, predicted_alice_item5_manual))


          Alice     User1     User2     User3     User4
Alice  1.000000  0.839181  0.606339  0.000000 -0.768095
User1  0.839181  1.000000  0.467707  0.489956 -0.900149
User2  0.606339  0.467707  1.000000 -0.161165 -0.466569
User3  0.000000  0.489956 -0.161165  1.000000 -0.641503
User4 -0.768095 -0.900149 -0.466569 -0.641503  1.000000
Predicted rating for Alice on Item5 (manual Pearson): 4.851676442821287
Part 2 (pandas) prediction: 4.871979899370592
Part 3 (manual) prediction: 4.851676442821287
Match: False
